# Lab 9- Deep Learning Model

This lab is meant to get you started in using Keras to design Deep Neural Networks. The goal here is to simply repeat your previous lab, but with DNNs.

Let's start with reading the data, like before:

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
        
filename="SUSY.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET", "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
RawNames=["l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi","MET", "MET_phi", "MET_rel", "axial_MET"]
FeatureNames=["M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]

df = pd.read_csv(filename, dtype='float64', names=VarNames)

Now lets define training and test samples. Note that DNNs take very long to train, so for testing purposes we will use only about 10% of the 5 million events in the training/validation sample. Once you get everything working, make the final version of your plots with the full sample. 

Also note that Keras had trouble with the Pandas tensors, so after doing all of the nice manipulation that Pandas enables, we convert the Tensor to a regular numpy tensor.

In [18]:
N_Max=550000
N_Train=500000

Train_Sample=df[:N_Train]
Test_Sample=df[N_Train:N_Max]

X_Train=np.array(Train_Sample[VarNames[1:]])
y_Train=np.array(Train_Sample["signal"])

X_Test=np.array(Test_Sample[VarNames[1:]])
y_Test=np.array(Test_Sample["signal"])


## Exercise 1

You will need to create several models and make sure they are properly trained. Write a function that takes this history and plots the values versus epoch. For every model that you train in the remainder of this lab, assess:

* Has you model's performance plateaued? If not train for more epochs. 
* Compare the performance on training versus test sample. Are you over training?

In [15]:
def plot_history(history):
 
    # Extract metrics from the history dictionary
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    # Set up the plot area
    plt.figure(figsize=(14, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'r--', label='Validation Accuracy')
    plt.title('Training vs. Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r--', label='Validation Loss')
    plt.title('Training vs. Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Display the plots
    plt.tight_layout()
    plt.show()

## Exercise 2

Following the original paper (see lab 7), make a comparison of the performance (using ROC curves and AUC) between models trained with raw, features, and raw+features data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Scale the data
scaler = StandardScaler()
X_Train_scaled = scaler.fit_transform(X_Train)
X_Test_scaled = scaler.transform(X_Test)

# Split Data into Raw, Features, and Raw Features
X_Train_raw = X_Train_scaled[:, :10]
X_Test_raw = X_Test_scaled[:, :10]

X_Train_feat = X_Train_scaled[:, 10:]
X_Test_feat = X_Test_scaled[:, 10:]

X_Train_all = X_Train_scaled
X_Test_all = X_Test_scaled


# Define the helper features
def plot_history(history, title_suffix=""):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 4))
    
    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'r--', label='Validation Accuracy')
    plt.title(f'Accuracy {title_suffix}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r--', label='Validation Loss')
    plt.title(f'Loss {title_suffix}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def train_susy_model(X_train_data, y_train_data, epochs=10):
    #Builds and trains a basic DNN with the given data.
    model = Sequential()
    model.add(Dense(64, input_dim=X_train_data.shape[1], activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    # Training (verbose=0 so it doesn't flood your notebook output)
    history = model.fit(X_train_data, y_train_data, epochs=epochs, 
                        batch_size=512, validation_split=0.2, verbose=0)
    
    return model, history

# Train Models and Plot Histories
model_raw, history_raw = train_susy_model(X_Train_raw, y_Train)
plot_history(history_raw, title_suffix="(Raw Data)")

model_feat, history_feat = train_susy_model(X_Train_feat, y_Train)
plot_history(history_feat, title_suffix="(Features Data)")

model_all, history_all = train_susy_model(X_Train_all, y_Train)
plot_history(history_all, title_suffix="(Raw + Features Data)")

# Evaluate Models and Plot ROC Curves 
# Predict probabilities for the test sets 
y_pred_raw = model_raw.predict(X_Test_raw, verbose=0).ravel()
y_pred_feat = model_feat.predict(X_Test_feat, verbose=0).ravel()
y_pred_all = model_all.predict(X_Test_all, verbose=0).ravel()

# Calculate FPR, TPR, and AUC
fpr_raw, tpr_raw, _ = roc_curve(y_Test, y_pred_raw)
auc_raw = auc(fpr_raw, tpr_raw)

fpr_feat, tpr_feat, _ = roc_curve(y_Test, y_pred_feat)
auc_feat = auc(fpr_feat, tpr_feat)

fpr_all, tpr_all, _ = roc_curve(y_Test, y_pred_all)
auc_all = auc(fpr_all, tpr_all)

# Plot everything together
plt.figure(figsize=(8, 8))
plt.plot(fpr_raw, tpr_raw, label=f'Raw Only (AUC = {auc_raw:.4f})')
plt.plot(fpr_feat, tpr_feat, label=f'Features Only (AUC = {auc_feat:.4f})')
plt.plot(fpr_all, tpr_all, label=f'Raw + Features (AUC = {auc_all:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')

plt.title('ROC Curve Comparison: SUSY Dataset')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

## Exercise 3

Design and implement at least 3 different DNN models. Train them and compare performance. You may try different architectures, loss functions, and optimizers to see if there is an effect.

In [ ]:
from tensorflow.keras.layers import BatchNormalization

# Define the 3 Experimental Models

def build_model_1_deep(input_shape):
    # Model 1: Deep and Narrow
    model = Sequential([
        Dense(32, input_dim=input_shape, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(8, activation='relu'),
        Dense(4, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def build_model_2_wide(input_shape):
    # Model 2: Wide and Shallow with RMSprop
    model = Sequential([
        Dense(256, input_dim=input_shape, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    # Using RMSprop optimizer instead of Adam
    model.compile(loss='binary_crossentropy', optimizer='rmsprop', metrics=['accuracy'])
    return model

def build_model_3_advanced(input_shape):
    # Model 3: Advanced with Batch Normalization and Nadam
    model = Sequential([
        Dense(64, input_dim=input_shape, activation='relu'),
        BatchNormalization(), # Normalizes activations of the previous layer
        Dropout(0.2),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='nadam', metrics=['accuracy'])
    return model

# Training the models
input_dim = X_Train_all.shape[1]
epochs = 10
batch_size = 512

model_1 = build_model_1_deep(input_dim)
history_1 = model_1.fit(X_Train_all, y_Train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)

model_2 = build_model_2_wide(input_dim)
history_2 = model_2.fit(X_Train_all, y_Train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)

model_3 = build_model_3_advanced(input_dim)
history_3 = model_3.fit(X_Train_all, y_Train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=0)

y_pred_1 = model_1.predict(X_Test_all, verbose=0).ravel()
y_pred_2 = model_2.predict(X_Test_all, verbose=0).ravel()
y_pred_3 = model_3.predict(X_Test_all, verbose=0).ravel()

# Calculate AUC 
fpr_1, tpr_1, _ = roc_curve(y_Test, y_pred_1)
auc_1 = auc(fpr_1, tpr_1)

fpr_2, tpr_2, _ = roc_curve(y_Test, y_pred_2)
auc_2 = auc(fpr_2, tpr_2)

fpr_3, tpr_3, _ = roc_curve(y_Test, y_pred_3)
auc_3 = auc(fpr_3, tpr_3)

# Plot ROC Curves 
plt.figure(figsize=(8, 8))
plt.plot(fpr_1, tpr_1, label=f'Model 1: Deep & Narrow (AUC = {auc_1:.4f})')
plt.plot(fpr_2, tpr_2, label=f'Model 2: Wide & Shallow (AUC = {auc_2:.4f})')
plt.plot(fpr_3, tpr_3, label=f'Model 3: Advanced/BatchNorm (AUC = {auc_3:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')

plt.title('ROC Curve Comparison: Architecture Experiments')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

## Exercise 4

Repeat exercise 4 from Lab 8, adding your best performing DNN as one of the models.  


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt

def max_significance(scores, y_true): 
    # Compute the maximum significance for a given set of predicted scores 
    thresholds = np.linspace(min(scores), max(scores), 100)  
    best_Z = 0 
    
    for t in thresholds: 
        S = np.sum((scores > t) & (y_true == 1))
        B = np.sum((scores > t) & (y_true == 0))
        if S + B > 0:
            Z = S / np.sqrt(S + B)
            best_Z = max(best_Z, Z)
            
    return best_Z

# Initialize Traditional Models 
models = { 
    "Logistic": LogisticRegression(max_iter=500),
    "LDA": LinearDiscriminantAnalysis(),
    "Naive Bayes": GaussianNB()
}

plt.figure(figsize=(9, 8)) 

for name, model in models.items(): 
    model.fit(X_Train_all, y_Train) 
    
    # Get predictions
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_Test_all)
    else:
        scores = model.predict_proba(X_Test_all)[:, 1]
        
    fpr, tpr, _ = roc_curve(y_Test, scores) 
    roc_auc = auc(fpr, tpr)
    
    # Calculate max significance
    z_max = max_significance(scores, y_Test)
    
    # Plot curve and add metrics to the legend
    plt.plot(fpr, tpr, label=f"{name} (AUC: {roc_auc:.3f}, Z_max: {z_max:.1f})")

dnn_scores = model_3.predict(X_Test_all, verbose=0).ravel()

fpr_dnn, tpr_dnn, _ = roc_curve(y_Test, dnn_scores)
roc_auc_dnn = auc(fpr_dnn, tpr_dnn)
z_max_dnn = max_significance(dnn_scores, y_Test)

plt.plot(fpr_dnn, tpr_dnn, label=f"Best DNN (AUC: {roc_auc_dnn:.3f}, Z_max: {z_max_dnn:.1f})", 
         linewidth=2.5, color='black')

# Finalize Plot
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.title('Model Comparison: Lab 8 Traditional ML vs. Best DNN')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right') 
plt.grid(alpha=0.3)
plt.show()